# Create and solve reduced economic dispatch problem
This notebook constructs and solves the reduced economic dispatch model.  

**Author:** Zhi Gao  
**Affiliation:** Utrecht University  
**Date:** June 2026  

**License:** MIT  
This notebook is provided for academic reproducibility only.  
Please cite the associated publication if you use this code.

## Table of Contents
1. Scenario configuration
1. Import pacakages & data loading
1. Construct and solve reduced+tightened problem
1. Construct and solve reduced problem
1. Validation and save metrics

In [ ]:
# ============================================================
# Scenario Definition
# ============================================================
YEAR = 1985 # or 2014

# Structural assumptions
WITH_STORAGE = True
WITH_TRANSMISSION = True

# Operational extensions
WITH_GRID_TARIFF = True
WITH_STORAGE_DISSIPATION = True

# ============================================================
# Surrogate Resolution Mapping
# ============================================================
SURROGATE_RESOLUTION = 'seasonal' # or any of below

dict_n_periods = {
    # String key -> number of clusters
    "seasonal": 4,      # 4 clusters per year
    "monthly": 12,       # 12 clusters per year
    "weekly": 59,        # ~59 weeks per year
    "daily": 365,        # 365 days per year
    "8h": 365 * 3,       # 3 clusters per day (8h each)
    "4h": 365 * 6,       # 6 clusters per day (4h each)
    "2h": 365 * 12,      # 12 clusters per day (2h each)
}

N_CLUSTERS = dict_n_periods[SURROGATE_RESOLUTION]

print(f"Scenario: Year={YEAR}, "
      f"Storage={WITH_STORAGE}, "
      f"Transmission={WITH_TRANSMISSION}, "
      f"Tariff={WITH_GRID_TARIFF}, "
      f"Dissipation={WITH_STORAGE_DISSIPATION}, "
      f"Surrogate resolution={SURROGATE_RESOLUTION}")

print(f"Scenario: Year={YEAR}, "
      f"Storage={WITH_STORAGE}, "
      f"Transmission={WITH_TRANSMISSION}, "
      f"Tariff={WITH_GRID_TARIFF}, "
      f"Dissipation={WITH_STORAGE_DISSIPATION}")

In [ ]:
# ============================================================
# Import packages
# ============================================================

# Optimization
import pyomo.environ as pyo

# Data handling
import pandas as pd
import numpy as np

# System & timing
import timeit
import os

# Utilities
from utils.solver_utils import solve_with_gurobi_logging
import pickle

# ============================================================
# Data Loading and Preprocessing
# ============================================================

# -------- Technology capacity --------
df_tech_capacity = pd.read_csv(
    "data/tech_capacity.csv",
    header=0, index_col=0
)

df_line_capacity = pd.read_csv(
    "data/line_capacity.csv",
    header=0, index_col=0
)

# -------- Apply scenario flags --------
if not WITH_STORAGE:
    df_tech_capacity["Battery"] *= 0
    df_tech_capacity["Battery_MWh"] *= 0
    df_tech_capacity["Hydro_Storage"] *= 0
    df_tech_capacity["Hydro_Storage_GWh"] *= 0
    
if not WITH_TRANSMISSION:
    df_line_capacity *= 0

# -------- Grid tariff --------
if WITH_GRID_TARIFF:
    df_grid_tarif = pd.read_csv(
        "data/grid_tarif.csv",
        index_col=0
    )

# ============================================================
# Technology Parameters
# ============================================================

# Storage characteristics
if WITH_STORAGE_DISSIPATION:
    dict_storage_hourly_self_dissipate_rate = {
        "Battery": 0.0001,
        "Hydro_Storage": 0.0001
    }

dict_storage_cycle_efficiency = {
    "Battery": 0.955,
    "Hydro_Storage": 0.875,
}

# Conversion efficiency
dict_conversion_efficiency = {
    "PEMEC": 0.585,
    "PEMFC": 0.5,
}

# Variable cost (Euro/MWh)
dict_variable_cost = {
    "Wind_Onshore": 2.06,
    "Wind_Offshore": 4.17,
    "Solar": 0.0,
    "CCGT": 79.0,
    "Battery": 0.1,
    "OCGT": 109.93,
    "Coal": 51.0,
    "Nuclear": 11.5,
    "E_ENS": 1000.0,
    "Run_of_River": 0.2,
    "Hydro_Storage": 0.2,
}

# ============================================================
# Sets Construction
# ============================================================

# Countries
list_country = np.unique(
    [asset[0:2] for asset in df_tech_capacity.index.values]
)

# Lines with positive capacity
list_line = [
    line for line in df_line_capacity.index.values
    if df_line_capacity.loc[line].values[0] > 0
]

# Country–technology output pairs
list_country_tech_output = []
for country in list_country:
    list_country_tech_output.append((country, "E_ENS"))
    for tech in [
        "Solar", "Wind_Offshore", "Wind_Onshore",
        "Nuclear", "Battery", "Coal", "CCGT",
        "OCGT", "Run_of_River", "Hydro_Storage"
    ]:
        if df_tech_capacity.loc[country, tech] > 0:
            list_country_tech_output.append((country, tech))

# Country–technology input (charging) pairs
list_country_tech_input = []
for country in list_country:
    for tech in ["Battery", "Hydro_Storage"]:
        if df_tech_capacity.loc[country, tech] > 0:
            list_country_tech_input.append((country, tech))

# ============================================================
# Time Series Profiles
# ============================================================

demand_profile = pd.read_csv(
    f"data/profiles/demand_profile_{YEAR}.csv",
    header=0, index_col=0
).round()

solar_profile = pd.read_csv(
    f"data/profiles/solar_profile_{YEAR}.csv",
    header=0, index_col=0
).round()

wind_offshore_profile = pd.read_csv(
    f"data/profiles/offshore_profile_{YEAR}.csv",
    header=0, index_col=0
).round()

wind_onshore_profile = pd.read_csv(
    f"data/profiles/onshore_profile_{YEAR}.csv",
    header=0, index_col=0
).round()

run_of_river_profile = pd.read_csv(
    f"data/profiles/run_of_river_profile_{YEAR}.csv",
    header=0, index_col=0
).round()

dict_tech_profile = {
    "Solar": solar_profile,
    "Wind_Offshore": wind_offshore_profile,
    "Wind_Onshore": wind_onshore_profile,
    "Run_of_River": run_of_river_profile,
}

print("✅ Data and sets successfully prepared.")

## Create and solve reduced + tightened problem

In [ ]:
# ============================================================
# Reduced Model (tightened)
# ============================================================

# -------- Experiment tracking --------
dict_experiment_performance = {
    "Count_Timestep": [],
    "Solver_Time_s": [],
    "Objective_Value": [],
}

# -------- Load surrogate bounds --------
base_path = f"surrogates_individual/{YEAR}/(s{int(WITH_STORAGE)}_t{int(WITH_TRANSMISSION)}_gt{int(WITH_GRID_TARIFF)}_sd{int(WITH_STORAGE_DISSIPATION)})"

print(f"Loading surrogate bounds from: {base_path}")
print(f"Clusters: {N_CLUSTERS}")

# Load dictionaries
with open(f"{base_path}/dict_vary_tech_output_{N_CLUSTERS}.p", "rb") as f:
    dict_vary_tech_output = pickle.load(f)

with open(f"{base_path}/dict_capacity_tech_output_{N_CLUSTERS}.p", "rb") as f:
    dict_capacity_tech_output = pickle.load(f)

with open(f"{base_path}/dict_lower_bound_{N_CLUSTERS}.p", "rb") as f:
    dict_vary_tech_output_lower_bound = pickle.load(f)

with open(f"{base_path}/dict_upper_bound_{N_CLUSTERS}.p", "rb") as f:
    dict_vary_tech_output_upper_bound = pickle.load(f)

print("✅ Surrogate bounds loaded")

# -------- Build reduced model --------
print("Building reduced model...")

list_country_tech_hour_output =  [(country,tech,hour) for country,tech in list_country_tech_output for hour in range(1,8761) if tech in dict_vary_tech_output[country,hour]]
start_time = timeit.default_timer()

model_reduced = pyo.ConcreteModel()
model_reduced.set_country = pyo.Set(initialize=list_country)
model_reduced.set_line = pyo.Set(initialize=list_line)
model_reduced.set_hour = pyo.Set(initialize=range(1,8761))
model_reduced.set_country_tech_hour_output = pyo.Set(initialize=list_country_tech_hour_output)
model_reduced.set_country_tech_input = pyo.Set(initialize=list_country_tech_input)

model_reduced.var_transmission = pyo.Var(model_reduced.set_line,model_reduced.set_hour,within=pyo.NonNegativeReals)

# Technology for every country
model_reduced.var_tech_output = pyo.Var(model_reduced.set_country_tech_hour_output,within=pyo.NonNegativeReals)
model_reduced.var_tech_input = pyo.Var(model_reduced.set_country_tech_input,model_reduced.set_hour,within=pyo.NonNegativeReals)
model_reduced.var_stored_energy = pyo.Var(model_reduced.set_country_tech_input,model_reduced.set_hour,within=pyo.NonNegativeReals)

def energy_balance_electricity(model_reduced,country,hour):
    return 0.9 * pyo.quicksum(model_reduced.var_transmission[line,hour] for line in model_reduced.set_line if line[3:5] == country)\
    - pyo.quicksum(model_reduced.var_transmission[line,hour] for line in model_reduced.set_line if line[0:2] == country)\
    + pyo.quicksum(model_reduced.var_tech_output[country,tech,hour] for tech in dict_vary_tech_output[country,hour])\
    - pyo.quicksum(model_reduced.var_tech_input[country_tech,hour] for country_tech in [country_tech for country_tech in list_country_tech_input if country_tech[0] == country])\
    == demand_profile.loc[hour,country] - pyo.quicksum(dict_tech_profile[tech].loc[hour,country] for tech in dict_capacity_tech_output[country,hour] if tech in ['Wind_Onshore','Wind_Offshore','Solar','Run_of_River'])\
    - pyo.quicksum(df_tech_capacity.loc[country,tech] for tech in dict_capacity_tech_output[country,hour] if tech in ['Nuclear','Coal','CCGT','OCGT'])

model_reduced.constraint_electricity_balance = pyo.Constraint(model_reduced.set_country,model_reduced.set_hour,rule=energy_balance_electricity)

def transmission_capacity_electricity(model_reduced,line,hour):
    return model_reduced.var_transmission[line,hour] <= df_line_capacity.loc[line,'Capacity']
model_reduced.constraint_transmission_capacity_electricity = pyo.Constraint(model_reduced.set_line,model_reduced.set_hour,rule=transmission_capacity_electricity)

def tech_output_capacity(model_reduced,country,tech,hour):
    if tech in ['Wind_Onshore','Wind_Offshore','Solar','Run_of_River']:
        return model_reduced.var_tech_output[country,tech,hour] <= dict_vary_tech_output_upper_bound[country,tech,hour] # dict_tech_profile[tech].loc[hour,country]
    elif tech in ['Nuclear','Coal','CCGT','OCGT','E_ENS']:
        return model_reduced.var_tech_output[country,tech,hour] <= dict_vary_tech_output_upper_bound[country,tech,hour] # df_tech_capacity.loc[country,tech] 
    elif tech in ['Battery','Hydro_Storage']:
        return model_reduced.var_tech_output[country,tech,hour] <= df_tech_capacity.loc[country,tech]
    else:
        print('Error here!' + tech)
model_reduced.constraint_tech_output_capacity = pyo.Constraint(model_reduced.set_country_tech_hour_output,\
                                                            rule=tech_output_capacity)

def tech_output_lower_bound(model_reduced,country,tech,hour):
    if dict_vary_tech_output_lower_bound[country,tech,hour] > 0:
        if tech in ['Wind_Onshore','Wind_Offshore','Solar','Run_of_River']:
            return model_reduced.var_tech_output[country,tech,hour] >= min(dict_tech_profile[tech].loc[hour,country],dict_vary_tech_output_lower_bound[country,tech,hour])
        elif tech in ['Nuclear','Coal','CCGT','OCGT','E_ENS']:
            return model_reduced.var_tech_output[country,tech,hour] >= dict_vary_tech_output_lower_bound[country,tech,hour]
        else:
            return pyo.Constraint.Skip
    else:
        return pyo.Constraint.Skip
model_reduced.constraint_tech_output_lower_bound = pyo.Constraint(model_reduced.set_country_tech_hour_output,\
                                                            rule=tech_output_lower_bound)


def tech_input_capacity(model_reduced,country,tech,hour):
    name_capacity = tech
    return model_reduced.var_tech_input[country,tech,hour] <= df_tech_capacity.loc[country,name_capacity]

model_reduced.constraint_tech_input_capacity = pyo.Constraint(model_reduced.set_country_tech_input,model_reduced.set_hour,\
                                                            rule=tech_input_capacity)

def stored_energy_capacity_uninvestable(model_reduced,country,tech,hour):
    if tech == 'Battery':
        return model_reduced.var_stored_energy[country,tech,hour] <= df_tech_capacity.loc[country,'Battery_MWh']
    else:
        return model_reduced.var_stored_energy[country,tech,hour] <= 1000 * df_tech_capacity.loc[country,tech+'_GWh']
model_reduced.constraint_stored_energy_capacity_uninvestable = pyo.Constraint(model_reduced.set_country_tech_input,model_reduced.set_hour,rule=stored_energy_capacity_uninvestable)

def storage_balance(model_reduced,country,tech,hour):
    if WITH_STORAGE_DISSIPATION:
        diss = dict_storage_hourly_self_dissipate_rate[tech]
    else:
        diss = 0.0

    if hour == 1:
        return  model_reduced.var_stored_energy[country,tech,hour] - (1 - diss) * model_reduced.var_stored_energy[country,tech,8760] == model_reduced.var_tech_input[country,tech,hour] - (1/dict_storage_cycle_efficiency[tech]) * model_reduced.var_tech_output[country,tech,hour]
    else:
        return  model_reduced.var_stored_energy[country,tech,hour] - (1 - diss) * model_reduced.var_stored_energy[country,tech,hour-1] == model_reduced.var_tech_input[country,tech,hour] - (1/dict_storage_cycle_efficiency[tech]) * model_reduced.var_tech_output[country,tech,hour]
model_reduced.constraint_storage_balance_no_inflow = pyo.Constraint(model_reduced.set_country_tech_input,model_reduced.set_hour,rule=storage_balance)

def total_cost(model_reduced):
    # Transmission tariff cost
    if WITH_GRID_TARIFF:
        trans_cost = pyo.quicksum(
            df_grid_tarif.loc[line].values[0]
            * model_reduced.var_transmission[line, hour]
            for line in model_reduced.set_line
            for hour in model_reduced.set_hour
        )
    else:
        trans_cost = 0
    return trans_cost + pyo.quicksum(dict_variable_cost[tech] * model_reduced.var_tech_output[country,tech,hour] for country,tech,hour in model_reduced.set_country_tech_hour_output)
model_reduced.obj = pyo.Objective(rule=total_cost,sense=pyo.minimize)

print(f'Finished objective in {round((timeit.default_timer()-start_time))} seconds.')
print("Reduced model size (Pyomo, pre-solve):")
print("  vars:", model_reduced.nvariables())
print("  cons:", model_reduced.nconstraints())

# ============================================================
# Solve Optimization problem
# ============================================================

logfile = (
    f"logs/reduced/{YEAR}/"
    f"reduced_{YEAR}_"
    f"(s{int(WITH_STORAGE)}_t{int(WITH_TRANSMISSION)})_"
    f"surrogate_{N_CLUSTERS}_tightened.log"
)

os.makedirs(os.path.dirname(logfile), exist_ok=True)

results, metrics = solve_with_gurobi_logging(
    pyomo_model=model_reduced,
    solver=pyo.SolverFactory("gurobi"),
    log_path=logfile,
    options={},  # Use defaults
)

In [ ]:
# -------- Calculate restored objective (including fixed capacity) --------
restored_objective = metrics['reported_lower_bound']
print(f"✅ Reduced model solved")
print(f"   Reduced problem objective: {restored_objective:,.1f}")
# Add fixed capacity costs
for hour in range(1, 8761):
    for country, tech in list_country_tech_output:
        if tech in dict_capacity_tech_output[country, hour]:
            if tech in ['Wind_Onshore','Wind_Offshore','Solar','Run_of_River']:
                restored_objective += (
                    dict_variable_cost[tech]
                    * dict_tech_profile[tech].loc[hour, country]
                )
            elif tech in ["Nuclear", "Coal", "CCGT", "OCGT"]:
                restored_objective += (
                    dict_variable_cost[tech]
                    * df_tech_capacity.loc[country, tech]
                )

# -------- Save metrics --------
df_metrics = pd.DataFrame(
    [
        {
            "model": "reduced",
            "year": YEAR,
            "n_clusters": N_CLUSTERS,
            "storage": WITH_STORAGE,
            "transmission": WITH_TRANSMISSION,
            "grid_tariff": WITH_GRID_TARIFF,
            "storage_dissipation": WITH_STORAGE_DISSIPATION,
            "restored_objective": restored_objective,
            **metrics,
        }
    ]
)

save_path = (
    f"results/"
    f"reduced_metrics_{YEAR}_"
    f"(s{int(WITH_STORAGE)}_t{int(WITH_TRANSMISSION)})_"
    f"surrogate_{N_CLUSTERS}_tightened.csv"
)

os.makedirs(os.path.dirname(save_path), exist_ok=True)
df_metrics.to_csv(save_path, index=False)

print(f"💾 Metrics saved to: {save_path}")

# -------- Update experiment performance --------
dict_experiment_performance["Count_Timestep"].append(N_CLUSTERS)
dict_experiment_performance["Solver_Time_s"].append(metrics["solver_wall_time_reported"])
dict_experiment_performance["Objective_Value"].append(restored_objective)

print("\n📊 Reduced Model Summary")
print(f"   Solver time: {metrics["solver_wall_time_reported"]:.1f}s")
print(f"   Memory delta: {metrics['rss_delta_GB']:.2f} GB")
print(f"   Reduced problem restored objective: {restored_objective:,.1f}")
df_metrics_dense = pd.read_csv(f'results/dense_metrics_{YEAR}_(s{int(WITH_STORAGE)}_t{int(WITH_TRANSMISSION)}_gt{int(WITH_GRID_TARIFF)}_sd{int(WITH_STORAGE_DISSIPATION)}).csv')
print(f"   Original dense problem objective: {df_metrics_dense['reported_lower_bound'][0]:,.1f}")

## Create and solve reduced problem (not tightened!)

In [ ]:
# ============================================================
# Reduced Model (variable reduced)
# ============================================================

# -------- Experiment tracking --------
dict_experiment_performance = {
    "Count_Timestep": [],
    "Solver_Time_s": [],
    "Objective_Value": [],
}

# -------- Load surrogate bounds --------
base_path = f"surrogates_individual/{YEAR}/(s{int(WITH_STORAGE)}_t{int(WITH_TRANSMISSION)})"

print(f"Loading surrogate bounds from: {base_path}")
print(f"Clusters: {N_CLUSTERS}")

# Load dictionaries
with open(f"{base_path}/dict_vary_tech_output_{N_CLUSTERS}.p", "rb") as f:
    dict_vary_tech_output = pickle.load(f)

with open(f"{base_path}/dict_capacity_tech_output_{N_CLUSTERS}.p", "rb") as f:
    dict_capacity_tech_output = pickle.load(f)

with open(f"{base_path}/dict_lower_bound_{N_CLUSTERS}.p", "rb") as f:
    dict_vary_tech_output_lower_bound = pickle.load(f)

with open(f"{base_path}/dict_upper_bound_{N_CLUSTERS}.p", "rb") as f:
    dict_vary_tech_output_upper_bound = pickle.load(f)

print("✅ Surrogate bounds loaded")

# -------- Build reduced model --------
print("Building reduced model...")

list_country_tech_hour_output =  [(country,tech,hour) for country,tech in list_country_tech_output for hour in range(1,8761) if tech in dict_vary_tech_output[country,hour]]
start_time = timeit.default_timer()

model_reduced = pyo.ConcreteModel()
model_reduced.set_country = pyo.Set(initialize=list_country)
model_reduced.set_line = pyo.Set(initialize=list_line)
model_reduced.set_hour = pyo.Set(initialize=range(1,8761))
model_reduced.set_country_tech_hour_output = pyo.Set(initialize=list_country_tech_hour_output)
model_reduced.set_country_tech_input = pyo.Set(initialize=list_country_tech_input)

model_reduced.var_transmission = pyo.Var(model_reduced.set_line,model_reduced.set_hour,within=pyo.NonNegativeReals)

# Technology for every country
model_reduced.var_tech_output = pyo.Var(model_reduced.set_country_tech_hour_output,within=pyo.NonNegativeReals)
model_reduced.var_tech_input = pyo.Var(model_reduced.set_country_tech_input,model_reduced.set_hour,within=pyo.NonNegativeReals)
model_reduced.var_stored_energy = pyo.Var(model_reduced.set_country_tech_input,model_reduced.set_hour,within=pyo.NonNegativeReals)

def energy_balance_electricity(model_reduced,country,hour):
    return 0.9 * pyo.quicksum(model_reduced.var_transmission[line,hour] for line in model_reduced.set_line if line[3:5] == country)\
    - pyo.quicksum(model_reduced.var_transmission[line,hour] for line in model_reduced.set_line if line[0:2] == country)\
    + pyo.quicksum(model_reduced.var_tech_output[country,tech,hour] for tech in dict_vary_tech_output[country,hour])\
    - pyo.quicksum(model_reduced.var_tech_input[country_tech,hour] for country_tech in [country_tech for country_tech in list_country_tech_input if country_tech[0] == country])\
    == demand_profile.loc[hour,country] - pyo.quicksum(dict_tech_profile[tech].loc[hour,country] for tech in dict_capacity_tech_output[country,hour] if tech in ['Wind_Onshore','Wind_Offshore','Solar','Run_of_River'])\
    - pyo.quicksum(df_tech_capacity.loc[country,tech] for tech in dict_capacity_tech_output[country,hour] if tech in ['Nuclear','Coal','CCGT','OCGT'])

model_reduced.constraint_electricity_balance = pyo.Constraint(model_reduced.set_country,model_reduced.set_hour,rule=energy_balance_electricity)

def transmission_capacity_electricity(model_reduced,line,hour):
    return model_reduced.var_transmission[line,hour] <= df_line_capacity.loc[line,'Capacity']
model_reduced.constraint_transmission_capacity_electricity = pyo.Constraint(model_reduced.set_line,model_reduced.set_hour,rule=transmission_capacity_electricity)

def tech_output_capacity(model_reduced,country,tech,hour):
    if tech in ['Wind_Onshore','Wind_Offshore','Solar','Run_of_River']:
        return model_reduced.var_tech_output[country,tech,hour] <= dict_tech_profile[tech].loc[hour,country]
    elif tech in ['Nuclear','Coal','CCGT','OCGT']:
        return model_reduced.var_tech_output[country,tech,hour] <= df_tech_capacity.loc[country,tech] 
    elif tech in ['Battery','Hydro_Storage']:
        return model_reduced.var_tech_output[country,tech,hour] <= df_tech_capacity.loc[country,tech]
    elif tech == 'E_ENS':
       return model_reduced.var_tech_output[country,tech,hour] <= demand_profile[country].max()
    else:
        print('Error here!' + tech)
model_reduced.constraint_tech_output_capacity = pyo.Constraint(model_reduced.set_country_tech_hour_output,\
                                                            rule=tech_output_capacity)


def tech_input_capacity(model_reduced,country,tech,hour):
    name_capacity = tech
    return model_reduced.var_tech_input[country,tech,hour] <= df_tech_capacity.loc[country,name_capacity]

model_reduced.constraint_tech_input_capacity = pyo.Constraint(model_reduced.set_country_tech_input,model_reduced.set_hour,\
                                                            rule=tech_input_capacity)

def stored_energy_capacity_uninvestable(model_reduced,country,tech,hour):
    if tech == 'Battery':
        return model_reduced.var_stored_energy[country,tech,hour] <= df_tech_capacity.loc[country,'Battery_MWh']
    else:
        return model_reduced.var_stored_energy[country,tech,hour] <= 1000 * df_tech_capacity.loc[country,tech+'_GWh']
model_reduced.constraint_stored_energy_capacity_uninvestable = pyo.Constraint(model_reduced.set_country_tech_input,model_reduced.set_hour,rule=stored_energy_capacity_uninvestable)

def storage_balance(model_reduced,country,tech,hour):
    if WITH_STORAGE_DISSIPATION:
        diss = dict_storage_hourly_self_dissipate_rate[tech]
    else:
        diss = 0.0

    if hour == 1:
        return  model_reduced.var_stored_energy[country,tech,hour] - (1 - diss) * model_reduced.var_stored_energy[country,tech,8760] == model_reduced.var_tech_input[country,tech,hour] - (1/dict_storage_cycle_efficiency[tech]) * model_reduced.var_tech_output[country,tech,hour]
    else:
        return  model_reduced.var_stored_energy[country,tech,hour] - (1 - diss) * model_reduced.var_stored_energy[country,tech,hour-1] == model_reduced.var_tech_input[country,tech,hour] - (1/dict_storage_cycle_efficiency[tech]) * model_reduced.var_tech_output[country,tech,hour]
model_reduced.constraint_storage_balance_no_inflow = pyo.Constraint(model_reduced.set_country_tech_input,model_reduced.set_hour,rule=storage_balance)

def total_cost(model_reduced):
    # Transmission tariff cost
    if WITH_GRID_TARIFF:
        trans_cost = pyo.quicksum(
            df_grid_tarif.loc[line].values[0]
            * model_reduced.var_transmission[line, hour]
            for line in model_reduced.set_line
            for hour in model_reduced.set_hour
        )
    else:
        trans_cost = 0
    return trans_cost + pyo.quicksum(dict_variable_cost[tech] * model_reduced.var_tech_output[country,tech,hour] for country,tech,hour in model_reduced.set_country_tech_hour_output)
model_reduced.obj = pyo.Objective(rule=total_cost,sense=pyo.minimize)

print(f'Finished objective in {round((timeit.default_timer()-start_time))} seconds.')
print("Reduced model size (Pyomo, pre-solve):")
print("  vars:", model_reduced.nvariables())
print("  cons:", model_reduced.nconstraints())

# ============================================================
# Solve Optimization problem
# ============================================================

logfile = (
    f"logs/reduced/{YEAR}/"
    f"reduced_{YEAR}_"
    f"(s{int(WITH_STORAGE)}_t{int(WITH_TRANSMISSION)})_"
    f"surrogate_{N_CLUSTERS}_reduced.log"
)

os.makedirs(os.path.dirname(logfile), exist_ok=True)

results, metrics = solve_with_gurobi_logging(
    pyomo_model=model_reduced,
    solver=pyo.SolverFactory("gurobi"),
    log_path=logfile,
    options={},  # Use defaults
)

In [ ]:
# -------- Calculate restored objective (including fixed capacity) --------
restored_objective = metrics['reported_lower_bound']
print(f"✅ Reduced model solved")
print(f"   Reduced problem objective: {restored_objective:,.1f}")
# Add fixed capacity costs
for hour in range(1, 8761):
    for country, tech in list_country_tech_output:
        if tech in dict_capacity_tech_output[country, hour]:
            if tech in ['Wind_Onshore','Wind_Offshore','Solar','Run_of_River']:
                restored_objective += (
                    dict_variable_cost[tech]
                    * dict_tech_profile[tech].loc[hour, country]
                )
            elif tech in ["Nuclear", "Coal", "CCGT", "OCGT"]:
                restored_objective += (
                    dict_variable_cost[tech]
                    * df_tech_capacity.loc[country, tech]
                )



# -------- Save metrics --------
df_metrics = pd.DataFrame(
    [
        {
            "model": "reduced",
            "year": YEAR,
            "n_clusters": N_CLUSTERS,
            "storage": WITH_STORAGE,
            "transmission": WITH_TRANSMISSION,
            "grid_tariff": WITH_GRID_TARIFF,
            "storage_dissipation": WITH_STORAGE_DISSIPATION,
            "restored_objective": restored_objective,
            **metrics,
        }
    ]
)

save_path = (
    f"results/"
    f"reduced_metrics_{YEAR}_"
    f"(s{int(WITH_STORAGE)}_t{int(WITH_TRANSMISSION)})_"
    f"surrogate_{N_CLUSTERS}_reduced.csv"
)

os.makedirs(os.path.dirname(save_path), exist_ok=True)
df_metrics.to_csv(save_path, index=False)

print(f"💾 Metrics saved to: {save_path}")

# -------- Update experiment performance --------
dict_experiment_performance["Count_Timestep"].append(N_CLUSTERS)
dict_experiment_performance["Solver_Time_s"].append(metrics["solver_wall_time_reported"])
dict_experiment_performance["Objective_Value"].append(restored_objective)

print("\n📊 Reduced Model Summary")
print(f"   Solver time: {metrics["solver_wall_time_reported"]:.1f}s")
print(f"   Memory delta: {metrics['rss_delta_GB']:.2f} GB")
print(f"   Reduced problem restored objective: {restored_objective:,.1f}")
df_metrics_dense = pd.read_csv(f'results/dense_metrics_{YEAR}_(s{int(WITH_STORAGE)}_t{int(WITH_TRANSMISSION)}_gt{int(WITH_GRID_TARIFF)}_sd{int(WITH_STORAGE_DISSIPATION)}).csv')
print(f"   Original dense problem objective: {df_metrics_dense['reported_lower_bound'][0]:,.1f}")